<a href="https://colab.research.google.com/github/pdf1802/f1-data-science/blob/main/notebooks/stint_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool 2 — Stint Optimizer
**Notebook:** `stint_optimizer.ipynb`  
**Depends on:** `lap_time_model.pkl`, `team_encoder.pkl`  
**What it does:** Sweeps every possible pit lap and finds the one that minimises total accumulated LapDelta across both stints.

> Module 1 answers: "how slow will this lap be?"  
> Tool 2 answers: "across a full race, which pit lap makes the total slowdown smallest?"  

No model training. No Monte Carlo. Pure deterministic sweep over Module 1 predictions.

In [1]:
!pip install fastf1 pandas numpy matplotlib plotly xgboost joblib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

CACHE_DIR = '/content/drive/MyDrive/f1_cache'
MODEL_DIR  = '/content/drive/MyDrive/f1_models'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

print(f'Cache dir : {CACHE_DIR}')
print(f'Model dir : {MODEL_DIR}')
print()
models_found = os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else []
print(f'Models found: {models_found if models_found else "(empty — check path)"}')

Mounted at /content/drive
Cache dir : /content/drive/MyDrive/f1_cache
Model dir : /content/drive/MyDrive/f1_models

Models found: ['compound_map.json', 'team_encoder.pkl', 'lap_time_model.pkl', 'overtaking_model.pkl', 'what_if_result.png']


## Imports & Models

We only need two `.pkl` files here — no overtaking model required.  
`lap_time_model.pkl` — predicts LapDelta given compound, tyre age, lap number, conditions.  
`team_encoder.pkl`   — converts "Ferrari" → integer.  

`compound_map.json`  — `{SOFT:0, MEDIUM:1, HARD:2}`, single source of truth.

> **Why no overtaking model?**  
> The Stint Optimizer asks about total time cost, not about position battles.  
> We assume clean air. The tool outputs the *theoretical* optimal, not a probabilistic one.  
> The Undercut Calculator (Tool 1) is where overtaking probability enters.

In [3]:
import os, warnings, json
import fastf1
import joblib
import numpy  as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
fastf1.Cache.enable_cache(CACHE_DIR)

# Load models
LAP_MODEL    = joblib.load(f'{MODEL_DIR}/lap_time_model.pkl')
TEAM_ENCODER = joblib.load(f'{MODEL_DIR}/team_encoder.pkl')

with open(f'{MODEL_DIR}/compound_map.json') as f:
    COMPOUND_MAP = json.load(f)
COMPOUND_NAMES = {v: k for k, v in COMPOUND_MAP.items()}

print('Models loaded')
print(f'Lap model   : {type(LAP_MODEL).__name__}')
print(f'Compounds   : {COMPOUND_MAP}')

Models loaded
Lap model   : XGBRegressor
Compounds   : {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2}


## Constants

`BASE_LAPTIME = 95.0s` — Bahrain 2024 average lap. LapDelta is *added on top*.  
`PIT_COST = 23.5s` — time lost vs staying out. Measured from FastF1 pit data.

> **Why add_noise=False here?**  
> In the Race Simulator we used `add_noise=True` because we wanted 1000 runs to diverge  
> into a *distribution* of outcomes. Here we want a clean, deterministic sweep.  
> We're asking "what is optimal" — noise would make the curve jagged and meaningless.  
> Think of it this way: noise is for *what will happen*, no-noise is for *what is true*.

In [4]:
BASE_LAPTIME     = 95.0
PIT_COST_DEFAULT = 23.5
PIT_COST_SC      = 17.0

SOFT, MEDIUM, HARD = 0, 1, 2

TEAM_COLOURS = {
    'Red Bull Racing' : '#3671C6',
    'Ferrari'         : '#E8002D',
    'Mercedes'        : '#27F4D2',
    'McLaren'         : '#FF8000',
    'Aston Martin'    : '#358C75',
    'Alpine'          : '#FF87BC',
    'Williams'        : '#64C4FF',
    'RB'              : '#6692FF',
    'Kick Sauber'     : '#52E252',
    'Haas F1 Team'    : '#B6BABD',
}

def encode_team(team_name: str) -> int:
    try:
        return int(TEAM_ENCODER.transform([team_name])[0])
    except:
        return 0

def encode_compound(name) -> int:
    if isinstance(name, int):
        return name
    return COMPOUND_MAP.get(name.upper(), 1)


def predict_lap_delta(compound: int, tyre_age: int, team_encoded: int,
                      lap_number: int, track_temp: float,
                      air_temp: float, rainfall: int = 0) -> float:
    """
    Single-row call to Module 1. Returns LapDelta in seconds.
    add_noise is always False here — we want a deterministic prediction.
    """
    row = pd.DataFrame([{
        'CompoundEncoded' : compound,
        'TyreLife'        : tyre_age,
        'TeamEncoded'     : team_encoded,
        'LapNumber'       : lap_number,
        'TrackTemp'       : track_temp,
        'AirTemp'         : air_temp,
        'RainfallEncoded' : rainfall,
    }])
    return float(LAP_MODEL.predict(row)[0])


print('Constants and helpers defined.')
print(f'PIT_COST_DEFAULT = {PIT_COST_DEFAULT}s  |  BASE_LAPTIME = {BASE_LAPTIME}s')

Constants and helpers defined.
PIT_COST_DEFAULT = 23.5s  |  BASE_LAPTIME = 95.0s


## `stint_cost()` — The Core Function

This is the most important function in the tool. It answers:

> "If I run compound X for `n_laps` laps starting with tyre_age=1, what is the total accumulated LapDelta?"

It loops lap by lap, calling Module 1 each time, and sums the deltas.

**Why loop and not vectorise?**  
`tyre_age` changes every lap, and `lap_number` changes every lap.  
We could build a DataFrame with all rows and call `.predict()` once — that's the  
vectorised version and it's ~10x faster. We write the loop first because it's  
readable. The vectorised version is in the Streamlit app (`shared.py`).

**Fuel effect:** Module 1 infers fuel burn from `LapNumber` alone. Lap 1 is heavy,  
lap 57 is light. We pass the *actual race lap number* (not the stint lap) so the  
model correctly accounts for fuel load across the whole race, not just within the stint.

In [12]:
def stint_cost(compound:int, n_laps:int, start_lap: int,
               team_encoded: int, config:dict, starting_tyre_age:int=1)->float:
    """
    Total accumulated LapDelta for one stint.

    Parameters
    ----------
    compound          : compound integer (SOFT=0, MEDIUM=1, HARD=2)
    n_laps            : how many laps this stint lasts
    start_lap         : which race lap the stint begins on (for fuel model)
    team_encoded      : integer from TEAM_ENCODER
    config            : race config dict (track_temp, air_temp, rainfall)
    starting_tyre_age : 1 for fresh tyres, >1 if starting on used rubber

    Returns
    -------
    Total LapDelta (seconds) accumulated across the stint.
    """

    total_delta=0.0
    tyre_age= starting_tyre_age

    for lap_offset in range(n_laps):
      race_lap=start_lap+ lap_offset
      delta= predict_lap_delta(
          compound=compound,
          tyre_age=tyre_age,
          team_encoded=team_encoded,
          lap_number=race_lap,
          track_temp=config['track_temp'],
          air_temp=config['air_temp'],
          rainfall=config.get('rainfall',0),
      )
      total_delta+=delta
      tyre_age+=1

    return total_delta

print('stint_cost() defined.')

# Quick sanity check: a soft on lap 1 vs lap 25
t_fresh = stint_cost(SOFT, 5, start_lap=1,  team_encoded=encode_team('Red Bull Racing'),
                     config={'track_temp': 37, 'air_temp': 29})
t_old   = stint_cost(SOFT, 5, start_lap=25, team_encoded=encode_team('Red Bull Racing'),
                     config={'track_temp': 37, 'air_temp': 29})
print(f'\nSoft tyre, 5 laps from lap 1  → total delta: {t_fresh:.3f}s')
print(f'Soft tyre, 5 laps from lap 25 → total delta: {t_old:.3f}s')
print(f'Difference: {t_old - t_fresh:.3f}s  ← degradation is real if positive')

stint_cost() defined.

Soft tyre, 5 laps from lap 1  → total delta: 11.558s
Soft tyre, 5 laps from lap 25 → total delta: 5.166s
Difference: -6.393s  ← degradation is real if positive


## `optimize_pit_lap()` — The Sweep

This is the search loop. For every candidate pit lap from lap 5 to lap N-5:

1. Compute stint 1 cost: run `compound_1` from lap 1 to `pit_lap`
2. Add `pit_cost` (23.5s under green)
3. Compute stint 2 cost: run `compound_2` from `pit_lap+1` to `total_laps`
4. Store the total

The function returns **two things**:
- The optimal pit lap (minimum of the total curve)
- The full cost array (every lap's total) — so we can plot the U-curve

> **Interview insight:** This is a 1D search over a convex-ish function.  
> In theory you could use golden-section search for speed, but with 45–50 candidate  
> laps it's fast enough to exhaustively enumerate. Exhaustive search is always  
> correct. Clever search is sometimes wrong. For this problem size, be correct.

In [13]:
def optimize_pit_lap(compound_1:int,compound_2:int, total_laps:int,
                     team_encoded:int, config:dict,
                     pit_cost: float = PIT_COST_DEFAULT,
                     earliest_pit:int = 5,
                     latest_pit: int = None) -> dict:
    """
    Sweep all candidate pit laps and return the optimal one + full cost curve.

    Returns
    -------
    dict with keys:
        optimal_lap   : int — lap with minimum total race time cost
        optimal_cost  : float — total LapDelta at optimal lap
        costs         : dict {pit_lap: total_cost} for all candidate laps
        stint1_costs  : dict {pit_lap: stint1_cost}
        stint2_costs  : dict {pit_lap: stint2_cost}
    """

    if latest_pit is None:
        latest_pit = total_laps - 5


    costs = {}
    stint1_costs = {}
    stint2_costs = {}

    for pit_lap in range(earliest_pit, latest_pit + 1):
      s1= stint_cost(compound=compound_1,n_laps = pit_lap, start_lap=1,
                     team_encoded=team_encoded, config=config)
      s2= stint_cost(compound=compound_2,n_laps = total_laps-pit_lap, start_lap=pit_lap + 1,
                     team_encoded=team_encoded, config=config)

      total= s1+pit_cost+s2
      costs[pit_lap] = total
      stint1_costs[pit_lap] = s1
      stint2_costs[pit_lap] = s2

    optimal_lap= min(costs, key=costs.get)
    optimal_cost= costs[optimal_lap]

    return{
        'optimal_lap': optimal_lap,
        'optimal_cost': optimal_cost,
        'costs': costs,
        'stint1_costs': stint1_costs,
        'stint2_costs': stint2_costs,
    }

print('optimize_pit_lap() defined.')

optimize_pit_lap() defined.


## Race Configuration

We validate on **Bahrain 2023 Round 1** — the same race used in Module 5.
This lets us cross-check our optimal window against what VER actually did (lap 21).

`pace_factor` modifies how fast this specific team's car is.
A factor of 1.0 = fastest car. 1.02 = 2% slower per lap ≈ +1.9s/lap.
For the optimizer this mostly shifts the absolute cost values, not the shape of the curve.
The optimal pit lap is driven by **tyre degradation**, not by car pace.

In [14]:
RACE_CONFIG = {
    'circuit'    : 'Bahrain 2023',
    'total_laps' : 57,
    'track_temp' : 37.0,
    'air_temp'   : 29.0,
    'rainfall'   : 0,
}

DRIVER = {
    'code'        : 'VER',
    'team'        : 'Red Bull Racing',
    'team_colour' : TEAM_COLOURS['Red Bull Racing'],
}
DRIVER['team_encoded'] = encode_team(DRIVER['team'])

print(f"Driver : {DRIVER['code']}  |  Team : {DRIVER['team']}")
print(f"Team encoded : {DRIVER['team_encoded']}")
print(f"Race : {RACE_CONFIG['circuit']}  |  {RACE_CONFIG['total_laps']} laps")
print(f"Track {RACE_CONFIG['track_temp']}°C  |  Air {RACE_CONFIG['air_temp']}°C")

Driver : VER  |  Team : Red Bull Racing
Team encoded : 11
Race : Bahrain 2023  |  57 laps
Track 37.0°C  |  Air 29.0°C


In [15]:
#Run the sweep
#Strategy A: MEDIUM -> HARD (what VER actually did in Bahrain 2023)
result_MH = optimize_pit_lap(
    compound_1=MEDIUM,
    compound_2=HARD,
    total_laps=RACE_CONFIG['total_laps'],
    team_encoded=DRIVER['team_encoded'],
    config=RACE_CONFIG,
)

#Strategy B: SOFT-> MEDIUM
result_SM = optimize_pit_lap(
    compound_1=SOFT,
    compound_2=MEDIUM,
    total_laps=RACE_CONFIG['total_laps'],
    team_encoded=DRIVER['team_encoded'],
    config=RACE_CONFIG,
)

print('=' * 55)
print(f"  Strategy A — MEDIUM → HARD")
print(f"  Optimal pit lap : Lap {result_MH['optimal_lap']}")
print(f"  Total LapDelta  : {result_MH['optimal_cost']:.2f}s")
print()
print(f"  Strategy B — SOFT → MEDIUM")
print(f"  Optimal pit lap : Lap {result_SM['optimal_lap']}")
print(f"  Total LapDelta  : {result_SM['optimal_cost']:.2f}s")
print()
delta_between = result_SM['optimal_cost'] - result_MH['optimal_cost']
winner = 'MEDIUM→HARD' if delta_between > 0 else 'SOFT→MEDIUM'
print(f"  Δ between strategies : {abs(delta_between):.2f}s  →  {winner} is faster")
print('=' * 55)

  Strategy A — MEDIUM → HARD
  Optimal pit lap : Lap 21
  Total LapDelta  : 112.45s

  Strategy B — SOFT → MEDIUM
  Optimal pit lap : Lap 22
  Total LapDelta  : 106.87s

  Δ between strategies : 5.58s  →  SOFT→MEDIUM is faster


## U-Curve Chart

The chart has three things:
1. **The U-curve** — total race LapDelta cost for each possible pit lap
2. **The optimal lap marker** — vertical dashed line at the minimum
3. **The actual pit lap marker** (if known) — to validate against reality

Two strategies are overlaid so you can see not just *when* to pit but *which compound
combination* costs less overall.

> **Reading the chart:** The width of the U-curve bottom tells you how sensitive  
> the strategy is. A flat bottom = you have a 6-lap window where it barely matters.  
> A sharp bottom = you must hit that exact lap or pay a real penalty.

In [16]:
laps_MH = sorted(result_MH['costs'].keys())
laps_SM = sorted(result_SM['costs'].keys())

costs_MH = [result_MH['costs'][l] for l in laps_MH]
costs_SM = [result_SM['costs'][l] for l in laps_SM]

fig = go.Figure()

# Strategy A — MEDIUM → HARD
fig.add_trace(go.Scatter(
    x=laps_MH, y=costs_MH,
    mode='lines',
    name='MEDIUM → HARD',
    line=dict(color='#FF8C00', width=3),
))

# Strategy B — SOFT → MEDIUM
fig.add_trace(go.Scatter(
    x=laps_SM, y=costs_SM,
    mode='lines',
    name='SOFT → MEDIUM',
    line=dict(color='#E8002D', width=3, dash='dash'),
))

# Optimal lap markers
for result, label, colour in [
    (result_MH, 'Opt MED→HRD', '#FF8C00'),
    (result_SM, 'Opt SFT→MED', '#E8002D'),
]:
    opt_lap  = result['optimal_lap']
    opt_cost = result['costs'][opt_lap]
    fig.add_vline(x=opt_lap, line_width=1.5, line_dash='dot', line_color=colour, opacity=0.7)
    fig.add_annotation(
        x=opt_lap, y=opt_cost,
        text=f'  {label}<br>  Lap {opt_lap}',
        showarrow=True, arrowhead=2, arrowcolor=colour,
        font=dict(color=colour, size=11),
        bgcolor='rgba(26,26,46,0.85)',
    )

# Actual VER pit lap (for validation)
ACTUAL_PIT_LAP = 21
fig.add_vline(
    x=ACTUAL_PIT_LAP,
    line_width=2, line_dash='longdash', line_color='#27F4D2',
    annotation_text=f'VER actual pit — Lap {ACTUAL_PIT_LAP}',
    annotation_font_color='#27F4D2',
    annotation_position='top right',
)

fig.update_layout(
    title=dict(
        text=f'Stint Optimizer — {DRIVER["code"]} | {RACE_CONFIG["circuit"]}',
        font=dict(color='white', size=16),
    ),
    xaxis=dict(
        title='Pit Lap', color='white', gridcolor='#333',
        title_font=dict(color='white'),
    ),
    yaxis=dict(
        title='Total Accumulated LapDelta (s)', color='white', gridcolor='#333',
        title_font=dict(color='white'),
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    legend=dict(font=dict(color='white'), bgcolor='#2a2a3e'),
    hovermode='x unified',
)

fig.show()

In [17]:
print(f"\n{'='*62}")
print(f"  STINT OPTIMIZER RESULTS — {DRIVER['code']} | {RACE_CONFIG['circuit']}")
print(f"{'='*62}")
print(f"  {'Strategy':<22} {'Opt Lap':>8} {'Stint 1 Δ':>12} {'Pit Cost':>10} {'Stint 2 Δ':>12} {'Total Δ':>10}")
print(f"  {'-'*60}")

for result, label in [(result_MH, 'MEDIUM → HARD'), (result_SM, 'SOFT → MEDIUM')]:
    opt = result['optimal_lap']
    s1  = result['stint1_costs'][opt]
    s2  = result['stint2_costs'][opt]
    tot = result['optimal_cost']
    print(f"  {label:<22} {opt:>8} {s1:>11.2f}s {PIT_COST_DEFAULT:>9.1f}s {s2:>11.2f}s {tot:>9.2f}s")

print(f"{'='*62}")
print(f"\n  VER actual pit: Lap {ACTUAL_PIT_LAP}")
opt_MH = result_MH['optimal_lap']
if abs(opt_MH - ACTUAL_PIT_LAP) <= 3:
    print(f"  ✅ Model optimal (Lap {opt_MH}) within 3 laps of reality → VALIDATED")
else:
    print(f"  ⚠️  Model optimal (Lap {opt_MH}) differs from reality by {abs(opt_MH - ACTUAL_PIT_LAP)} laps → investigate")


  STINT OPTIMIZER RESULTS — VER | Bahrain 2023
  Strategy                Opt Lap    Stint 1 Δ   Pit Cost    Stint 2 Δ    Total Δ
  ------------------------------------------------------------
  MEDIUM → HARD                21       51.74s      23.5s       37.21s    112.45s
  SOFT → MEDIUM                22       40.69s      23.5s       42.68s    106.87s

  VER actual pit: Lap 21
  ✅ Model optimal (Lap 21) within 3 laps of reality → VALIDATED


Tool 2 validates exactly against Bahrain 2023. The Soft→Medium strategy is 5.58s faster in pure accumulated LapDelta, but MEDIUM→HARD offers more consistent pace. The model reveals the real cost of each strategy choice, not just which one wins.